# UBC-OCEAN – Ovarian Carcinoma Subtype Classification

Classify ovarian cancer histology images into five subtypes: HGSC, EC, CC, LGSC and MC, plus "Other" for outliers.
Metric: balanced accuracy. Late submissions, scored after the competition closed.

**Data:**
- Train has 538 images: 513 whole-slide images (WSI, 20x, up to ~100k px a side) and 25 tissue microarray cores (TMA, 40x, ~3-4k px).
- The hidden test has ~2000 images, mostly TMAs from other hospitals. It also contains an "Other" class that never appears in train.
- The raw train set is 790 GB of PNG. That makes the cross-hospital gap the real problem, more than the model.

**Pipeline:**
1. **Tiling:** each WSI becomes 48 random 7×7-tile blocks of 224 px tiles. Each block is about one TMA core, so it can stand in for a TMA during training.
   TMAs are downscaled by 0.5 to match the WSI's 20x scale, and all their tissue tiles are kept.
2. **Features:** a frozen pathology foundation model (Phikon-v2, ViT-L) embeds every tile. CLS token, fp16.
3. **External data:** 2159 labelled TMA cores from four outside sources, including 738 "Other" cores. Each source is rescaled to the train scale.
4. **MIL head:** a gated-attention ABMIL head classifies each bag of tile features. 5-fold CV, 3 seeds.
5. **Kaggle inference:** the same tiling code, then the encoder, then the ABMIL ensemble. Runs offline on a T4.

**Results** (balanced accuracy):

| model | OOF all | OOF WSI | OOF pseudo-TMA | OOF external TMAs | public LB | private LB |
|---|---|---|---|---|---|---|
| base: competition data only | 0.861 | 0.862 | 0.763 | ~0.5 (never seen) | 0.57 | **0.58** |
| ext_all: + 2159 external TMAs, 6 classes | 0.844 | 0.845 | 0.743 | 0.75 | 0.57 | 0.56 |
| base + ext_all, 3 seeds each, 50/50 | 0.867 | 0.866 | | | | |

The external TMAs lift accuracy on other-hospital cores from ~0.5 to ~0.75, and they teach the model an "Other" class (93% recall, 2% false positives).
That did not move the leaderboard, though: both models score 0.57 on the public LB, and the gap is within noise.
For reference, public LB silver starts at 0.49 and gold at about 0.60.

## 0. Setup and data

Python 3.11. The scripts never hold the whole train set: `prep.py` downloads one PNG at a time, tiles it and deletes it.
- **Encoders:** Phikon-v2 needs a Hugging Face login.
  UNI (ViT-L, MahmoodLab) was the first choice, but access was denied, so `feats.py` also accepts a timm UNI state dict.
- **External data:** the TMA collection comes from the 10th-place solution (Kaggle dataset `gunesevitan/ubc-ocean-dataset`). Its license is listed as "unknown".

In [ ]:
!pip install -q kaggle pyvips opencv-python-headless pandas scikit-learn timm transformers torch huggingface_hub
!kaggle competitions download -c ubc-ocean -f train.csv -p data; unzip -oq -d data "data/*.zip"; rm -f data/*.zip
!huggingface-cli download owkin/phikon-v2 --local-dir weights/phikon-v2   # after `huggingface-cli login`
!huggingface-cli download owkin/phikon --local-dir weights/phikon
!mkdir -p ext/gv && kaggle datasets download gunesevitan/ubc-ocean-dataset -p ext/gv && cd ext/gv && unzip -oq "*.zip"

## 1. Tiling

- **WSIs:** pyvips decodes the PNG in horizontal strips of one block height (1568 px), so RAM stays at about 1 GB even for 100k px slides.
  - A block counts as tissue when at least half its tiles are tissue (HSV saturation > 25 and value > 40).
  - A reservoir sample keeps 48 such blocks, uniformly at random, in one pass.
  - Each block keeps only its tiles that are at least 50% tissue.
- **TMAs** are 40x, so they are downscaled by 0.5 and every tissue tile is kept, as one block.
- **Pale slides:** two train WSIs are pale enough that nothing passes the tissue mask. The tiler retries once with saturation > 8.
- **Storage:** tiles are written as JPEG (quality 90), about 7× smaller than raw pixels. The Kaggle inference does the same JPEG round-trip, so train and test pixels match.
- **Throughput:** the download is the bottleneck. Several CPU machines each ran 4 workers on their own `--ids` subset.

In [ ]:
%%writefile prep.py
"""Kaggle PNG -> tissue tiles (JPEG) at a common ~20x scale, then delete the PNG (raw train is 790 GB).

WSI (20x): keep K random 7x7-tile blocks (1568 px, about one TMA core) -> each block can stand in for a TMA at train time.
TMA (40x): downscale 0.5 -> all tissue tiles, one block.
Output: OUT/tiles/{id}/{block:02d}_{y}_{x}.jpg (y, x at 20x scale). A finished slide has OUT/tiles/{id}/done.

usage: python prep.py OUT [--workers 4] [--blocks 48] [--ids 91,4]
       python prep.py OUT --png some.png --tma 0   # tile a local file (no download/delete)
"""
import argparse
import heapq
import shutil
import tempfile
import zipfile
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import pyvips

T, NB = 224, 7  # tile px, tiles per block side
B = T * NB


def tissue(rgb, smin=25):
    """Per-tile tissue fraction for an (h, w, 3) array with h, w multiples of T. Background is white glass or black padding."""
    small = rgb[::8, ::8]
    hsv = cv2.cvtColor(np.ascontiguousarray(small), cv2.COLOR_RGB2HSV)
    m = (hsv[..., 1] > smin) & (hsv[..., 2] > 40)
    s = T // 8
    return m.reshape(m.shape[0] // s, s, m.shape[1] // s, s).mean((1, 3))


def jpg(a):
    return cv2.imencode(".jpg", cv2.cvtColor(a, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 90])[1].tobytes()


def tiles_of(block, y0, x0, smin, min_frac=0.5):
    """Tissue tiles of one block -> [(y, x, jpeg)]."""
    f = tissue(block, smin)
    return [(y0 + i * T, x0 + j * T, jpg(block[i * T:(i + 1) * T, j * T:(j + 1) * T]))
            for i, j in zip(*np.nonzero(f >= min_frac))]


def tile_image(path, tma, k, seed, smin=25):
    """-> [[(y, x, jpeg)] per block]. Nothing passes the mask (pale stain: 2 train WSIs) -> retry with a looser one."""
    blocks = _tile_image(path, tma, k, seed, smin)
    return blocks if any(blocks) or smin <= 8 else tile_image(path, tma, k, seed, 8)


def _tile_image(path, tma, k, seed, smin):
    im = pyvips.Image.new_from_file(str(path), access="sequential")
    im = im[:3] if im.bands > 3 else im
    if tma:
        a = im.resize(0.5).numpy()
        h, w = a.shape[0] // T * T, a.shape[1] // T * T
        return [tiles_of(a[:h, :w], 0, 0, smin)]
    rng = np.random.default_rng(seed)
    heap = []  # (-key, n, tiles): keeps the k blocks with the smallest random keys = uniform sample
    n = 0
    for y0 in range(0, im.height - B + 1, B):  # strips top->bottom: PNG decodes sequentially, RAM stays ~1 GB
        strip = im.crop(0, y0, im.width // B * B, B).numpy()
        f = tissue(strip, smin)
        for bx in range(strip.shape[1] // B):
            if (f[:, bx * NB:(bx + 1) * NB] >= 0.5).mean() < 0.5:
                continue
            key = rng.random()
            if len(heap) < k or key < -heap[0][0]:
                t = tiles_of(strip[:, bx * B:(bx + 1) * B], y0, bx * B, smin)
                item = (-key, n, t)
                heapq.heappush(heap, item) if len(heap) < k else heapq.heapreplace(heap, item)
                n += 1
    return [t for *_, t in sorted(heap, key=lambda h: (h[2][0][0], h[2][0][1]))]


def save(blocks, d):
    tmp = d.with_name(d.name + ".tmp")
    shutil.rmtree(tmp, ignore_errors=True)
    tmp.mkdir(parents=True)
    for b, ts in enumerate(blocks):
        for y, x, data in ts:
            (tmp / f"{b:02d}_{y}_{x}.jpg").write_bytes(data)
    (tmp / "done").touch()
    shutil.rmtree(d, ignore_errors=True)
    tmp.rename(d)
    return sum(map(len, blocks))


def download(api, image_id, tmp):
    api.competition_download_file("ubc-ocean", f"train_images/{image_id}.png", path=tmp, quiet=True)
    p = next(f for f in Path(tmp).rglob("*") if f.is_file())  # CLI may nest it under train_images/
    if p.suffix == ".zip":
        with zipfile.ZipFile(p) as z:
            z.extractall(tmp)
        p.unlink()
        p = next(Path(tmp).rglob("*.png"))
    return p


def job(image_id, tma, out, k):
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    with tempfile.TemporaryDirectory(dir=out) as tmp:  # PNG deleted with the dir, even on error
        p = download(api, image_id, tmp)
        gb = p.stat().st_size / 1e9
        n = save(tile_image(p, tma, k, image_id), out / "tiles" / str(image_id))
    return image_id, gb, n


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("out")
    ap.add_argument("--csv", default="data/train.csv")
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--blocks", type=int, default=48)
    ap.add_argument("--ids", default="")
    ap.add_argument("--png", help="tile one local PNG instead (debug / inference parity)")
    ap.add_argument("--tma", type=int, default=0)
    a = ap.parse_args()
    out = Path(a.out)
    if a.png:
        print(save(tile_image(a.png, a.tma, a.blocks, 0), out / "tiles" / Path(a.png).stem), "tiles")
        return
    df = pd.read_csv(a.csv)
    if a.ids:
        df = df[df.image_id.astype(str).isin(a.ids.split(","))]
    todo = [r for r in df.itertuples() if not (out / "tiles" / str(r.image_id) / "done").exists()]
    print(f"{len(todo)} / {len(df)} to do", flush=True)
    with ProcessPoolExecutor(a.workers) as ex:
        futs = {ex.submit(job, r.image_id, r.is_tma, out, a.blocks): r.image_id for r in todo}
        for i, f in enumerate(as_completed(futs), 1):
            try:
                iid, gb, n = f.result()
                print(f"[{i}/{len(todo)}] {iid}: {gb:.2f} GB png -> {n} tiles", flush=True)
            except Exception as e:  # leave it undone; a rerun retries it
                print(f"[{i}/{len(todo)}] {futs[f]} FAILED: {e!r}", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
!nohup python prep.py . --workers 4 > prep.log 2>&1 &   # writes tiles/{image_id}/; rerun to retry failures

## 2. Tile features

- **Encoders:** frozen Phikon-v2 (ViT-L/16, 1024-d) and Phikon (ViT-B/16, 768-d). Both use the CLS token and ImageNet mean/std normalisation, and output fp16.
- **Views:** view 0 is the plain tiles.
  View 1 applies a random colour shift per slide (hue, saturation, contrast, brightness) plus a random flip per tile. It mimics another hospital's stain and is used only as training augmentation.
- **Sharding:** `--shard i/n` splits slides across GPU machines. Phikon on an RTX 3050 takes about 10 s per WSI (~2000 tiles).

In [ ]:
%%writefile feats.py
"""Tiles -> frozen pathology-encoder features, fp16. View 0 = plain tiles; views 1.. = stain/flip-jittered copies (train-time aug).
One view per run so plain features for every slide land first. The encoder is named by its weights dir
(weights/phikon-v2, weights/phikon: HF checkpoints, CLS token; weights/UNI: timm state dict).

Output: feats/{encoder}/v{view}/{id}.npz with f (n, d) fp16, b (n,) block id, yx (n, 2).
usage: python feats.py TILES [--weights weights/phikon-v2] [--view 0] [--shard 0/3]
"""
import argparse
import zlib
from pathlib import Path

import cv2
import numpy as np
import timm
import torch

MEAN = np.array([0.485, 0.456, 0.406], np.float32) * 255
STD = np.array([0.229, 0.224, 0.225], np.float32) * 255


class CLS(torch.nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, x):
        return self.m(pixel_values=x).last_hidden_state[:, 0]


def encoder(wdir):
    wdir = Path(wdir)
    if wdir.name == "UNI":
        m = timm.create_model("vit_large_patch16_224", img_size=224, patch_size=16, init_values=1e-5,
                              num_classes=0, dynamic_img_size=True)
        m.load_state_dict(torch.load(wdir / "pytorch_model.bin", map_location="cpu"))
        return m
    from transformers import AutoModel
    return CLS(AutoModel.from_pretrained(wdir))


def stain(rng):
    """One colour shift per slide+view (hue, sat scale, contrast, brightness): mimics another hospital's staining.
    Per-tile jitter would mostly average out under attention pooling."""
    return rng.uniform(-10, 10), rng.uniform(0.6, 1.4), rng.uniform(0.7, 1.3), rng.uniform(-30, 20)


def jitter(rgb, st, rng):
    """Slide-level stain shift st + per-tile random dihedral flip."""
    dh, ds, c, b = st
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[..., 0] = (hsv[..., 0] + dh) % 180
    hsv[..., 1] *= ds
    rgb = cv2.cvtColor(np.clip(hsv, 0, 255).astype(np.uint8), cv2.COLOR_HSV2RGB).astype(np.float32)
    rgb = (rgb - 128) * c + 128 + b
    rgb = np.rot90(rgb, rng.integers(4))
    return np.clip(rgb[:, ::-1] if rng.random() < 0.5 else rgb, 0, 255)


class Tiles(torch.utils.data.Dataset):
    def __init__(self, files, view, slide):
        self.files, self.view = files, view
        self.st = stain(np.random.default_rng((slide, view)))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        x = cv2.cvtColor(cv2.imread(str(self.files[i])), cv2.COLOR_BGR2RGB)
        if self.view:
            x = jitter(x, self.st, np.random.default_rng((i, self.view)))
        return torch.from_numpy(((x - MEAN) / STD).transpose(2, 0, 1).copy())


@torch.no_grad()
def embed(model, files, view, slide, bs, workers):
    dl = torch.utils.data.DataLoader(Tiles(files, view, slide), bs, num_workers=workers, pin_memory=True)
    out = []
    for x in dl:
        with torch.autocast("cuda", dtype=torch.float16):
            out.append(model(x.cuda(non_blocking=True)).half().cpu())
    return torch.cat(out).numpy()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("tiles")
    ap.add_argument("--weights", default="weights/phikon-v2")
    ap.add_argument("--view", type=int, default=0)
    ap.add_argument("--shard", default="0/1")
    ap.add_argument("--bs", type=int, default=64)  # 128 hits the 6 GB limit with ViT-L
    ap.add_argument("--workers", type=int, default=12)
    a = ap.parse_args()
    k, n = map(int, a.shard.split("/"))
    out = Path("feats") / Path(a.weights).name / f"v{a.view}"
    out.mkdir(parents=True, exist_ok=True)
    slides = sorted(d for d in Path(a.tiles).iterdir() if (d / "done").exists())[k::n]
    model = encoder(a.weights).cuda().eval()
    for i, d in enumerate(slides):
        if (out / f"{d.name}.npz").exists():
            continue
        files = sorted(d.glob("*.jpg"))
        if not files:  # prep's tissue mask found nothing (2 pale train WSIs); train.py skips slides without npz
            continue
        parts = [p.stem.split("_") for p in files]
        f = embed(model, files, a.view, zlib.crc32(d.name.encode()), a.bs, a.workers)  # ext ids are not ints
        np.savez(out / f"{d.name}.tmp.npz", f=f, b=np.array([int(p[0]) for p in parts]),
                 yx=np.array([[int(p[1]), int(p[2])] for p in parts]))
        (out / f"{d.name}.tmp.npz").rename(out / f"{d.name}.npz")
        print(f"[{i + 1}/{len(slides)}] {d.name}: {f.shape}", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
!python feats.py tiles --view 0          # plain features, needed for everything else
!python feats.py tiles --view 1          # stain-jittered copy (train-time augmentation)
!python feats.py tiles_ext --view 0      # external TMAs (section 3)
!python feats.py tiles --weights weights/phikon --view 0   # second encoder

## 3. External TMAs

The hidden test is mostly TMAs from hospitals not in train. Train has only 25 TMAs, so outside cores are the most direct way to cover that gap.

**Sources** (2171 cores; 2159 have tissue after tiling):

| source | stain | scale | CC | EC | HGSC | LGSC | MC | Other |
|---|---|---|---|---|---|---|---|---|
| tissuearray.com (`tad`) | H&E | ×2 | 24 | 39 | 348 | 140 | 100 | 570 |
| usbiolab.com (`usb`) | H&E | ×2 | 29 | 40 | 124 | 89 | 68 | 90 |
| Human Protein Atlas (`hpa`) | IHC | ×1 | | 155 | 25 | 63 | 133 | |
| Stanford TMA DB (`stan`) | IHC | ×1 | 4 | 11 | 37 | | 4 | 78 |

- **Scales:** chosen by matching median nucleus size (hematoxylin blobs) to the train tiles, then checked by eye. `tad` and `usb` are low-resolution images of whole cores, so they are upsampled by 2.
- **Labels:** Stanford diagnoses are mapped to the five subtypes. Non-carcinomas (fibroma, lymphoma, dysgerminoma, ...) become "Other".
- **IHC sources:** `hpa` and `stan` are immunohistochemistry stains, not H&E. Adding them on top of the two H&E sources changed OOF accuracy by less than 0.01.

In [ ]:
%%writefile ext.py
"""External TMAs (from the 10th-place dataset gunesevitan/ubc-ocean-dataset) -> tiles_ext/{src}_{n}/ + data/ext.csv.

Each source is rescaled so nuclei match train TMA tiles (40x TMA -> 0.5 = 20x), then tiled like a train TMA.
Scales: median nucleus area (HED hematoxylin blobs) vs train tiles, then eyeballed; tissuearray/usbiolab are
low-res screenshots of whole cores. hpa + stanford are IHC (DAB) stains, not H&E: train.py --ext picks sources.
kztymsrjx9 is skipped: mixed magnifications in small photos.
usage: python ext.py ext/gv/datasets [--workers 12]
"""
import argparse
import re
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import pandas as pd
import pyvips

from prep import T, save, tiles_of

SRC = {  # name: (glob, scale)
    "hpa": ("human_protein_atlas/images/*", 1.0),
    "stan": ("stanford_tissue_microarray_database_ovarian_cancer_tma/images/*", 1.0),
    "tad": ("tissuearraycom/images/*", 2.0),
    "usb": ("usbiolabcom/images/*", 2.0),
}
STANFORD = {"carcinoma papillary serous": "HGSC", "carcinoma adeno": "HGSC", "carcinoma endometrioid": "EC",
            "carcinoma clear cell": "CC", "carcinoma mucinous": "MC", "carcinoma adeno mucinous": "MC"}


def label(src, p):
    if src == "stan":
        dx = re.sub(r"^\d+_\d+_ovary_", "", p.stem)
        return STANFORD.get(dx, "Other")  # fibroma, lymphoma, dysgerminoma
    return p.stem.split("_")[-1]


def job(src, p, scale, out):
    a = pyvips.Image.new_from_file(str(p))
    a = (a[:3] if a.bands > 3 else a).resize(scale).numpy()
    h, w = a.shape[0] // T * T, a.shape[1] // T * T
    blocks = [tiles_of(a[:h, :w], 0, 0, 25)] if h and w else [[]]
    if not blocks[0]:
        blocks = [tiles_of(a[:h, :w], 0, 0, 8)]
    return save(blocks, out)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("root")
    ap.add_argument("--workers", type=int, default=12)
    a = ap.parse_args()
    rows, futs = [], []
    with ProcessPoolExecutor(a.workers) as ex:
        for src, (g, scale) in SRC.items():
            for p in sorted(Path(a.root).glob(g)):
                iid = src + "_" + ("_".join(p.name.split("_")[:2]) if src == "stan" else p.stem)  # stem: n_LABEL (n not unique)
                rows.append({"image_id": iid, "label": label(src, p), "src": src})
                out = Path("tiles_ext") / iid
                if not (out / "done").exists():
                    futs.append(ex.submit(job, src, p, scale, out))
        n = [f.result() for f in futs]
    df = pd.DataFrame(rows)
    df.to_csv("data/ext.csv", index=False)
    print(df.groupby(["src", "label"]).size().unstack(fill_value=0), f"\ntiled {len(n)}, tiles min/median {min(n, default=0)}/{sorted(n)[len(n) // 2] if n else 0}")


if __name__ == "__main__":
    main()

In [ ]:
!python ext.py ext/gv/datasets --workers 12   # tiles_ext/{src}_{id}/ + data/ext.csv

## 4. Gated-attention MIL

- **Model:** a gated-attention ABMIL head: Linear(d → 256), GELU and dropout, then attention pooling and a 6-way linear head.
- **Training bags (per WSI step):**
  - With probability 0.5 the bag is one random 7×7 block, a pseudo-TMA. Otherwise it is up to 512 random tiles from the whole slide.
  - 10% of tiles are then dropped.
  - The feature view is picked at random from `--views`.
- **Loss:** class-weighted cross entropy (inverse frequency) with label smoothing 0.1 and `reduction="sum"`. Real TMAs get weight 2.
  - A weighted *mean* over a bag of size 1 divides the class weight back out.
  - Classes absent from a fold get weight 0. Otherwise label smoothing alone pulls predictions toward "Other": one run collapsed to 0.04 balanced accuracy that way.
- **Optimiser:** AdamW, lr 2e-4, OneCycle schedule, 40 epochs, one bag per step. Gradient accumulation over 16 bags starved the optimiser: 0.26 vs 0.73 balanced accuracy.
- **Validation:**
  - 5-fold, stratified on label × is_tma.
  - External cores get their own 80/20 split per fold, which gives an other-hospital TMA estimate.
  - Pseudo-TMA scores every block of a validation WSI as its own bag.
- **Speed:** CPU only, 4 threads. A 5-fold run takes about 6 minutes without external data.

**What did not help** (OOF, 3 seeds):

| run | OOF all | ext TMAs |
|---|---|---|
| ext_all (plain view) | 0.842 | ~0.74 |
| + stain view 1 (`--views 0,1`) | 0.839 | ~0.74 |
| external weight 2 (`--ext_w 2`, 1 seed) | 0.841 | ~0.74 |
| entropy threshold for Other | | flags 16% of in-distribution WSIs at a useful cutoff |

In [ ]:
%%writefile train.py
"""Gated-attention MIL on frozen tile features, 5-fold CV stratified on label x is_tma.

Train bags per WSI: either one random 7x7 block (pseudo-TMA) or a random subset of all tiles; feature view picked at random.
Val reports balanced accuracy on WSI (all tiles), real TMA, and pseudo-TMA (every block of val WSIs as its own bag).

usage: python train.py --name abmil [--feats feats --seeds 0,1,2 --epochs 40 ...]
"""
import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold

CLASSES = ["CC", "EC", "HGSC", "LGSC", "MC", "Other"]  # Other only from external TMAs (--ext)


class ABMIL(nn.Module):
    def __init__(self, d_in=1024, d=256, n_cls=6, drop=0.25):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(d_in, d), nn.GELU(), nn.Dropout(drop))
        self.a, self.b, self.w = nn.Linear(d, 128), nn.Linear(d, 128), nn.Linear(128, 1)
        self.head = nn.Linear(d, n_cls)

    def forward(self, x):  # (n, d_in) -> (n_cls,)
        h = self.fc(x)
        att = torch.softmax(self.w(torch.tanh(self.a(h)) * torch.sigmoid(self.b(h))), 0)
        return self.head((att * h).sum(0))


def load(feats, df, views):
    data = {}
    for r in df.itertuples():
        zs = [np.load(Path(feats) / f"v{v}" / f"{r.image_id}.npz") for v in views]
        data[r.image_id] = (torch.from_numpy(np.stack([z["f"] for z in zs])), zs[0]["b"])
    return data


def bags(f, b, tma, view=0):
    """Eval bags: (whole slide, [one per block])."""
    x = f[view].float()
    return x, ([x] if tma else [x[b == k] for k in np.unique(b)])


def run_fold(args, data, tr, va, seed):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = ABMIL(d_in=data[tr.image_id.iloc[0]][0].shape[-1], drop=args.drop)
    opt = torch.optim.AdamW(model.parameters(), args.lr, weight_decay=args.wd)
    steps = args.epochs * len(tr) // args.accum
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, args.lr, total_steps=steps + 1, pct_start=0.1)
    counts = np.bincount(tr.y.astype(int), minlength=len(CLASSES))
    cw = np.where(counts > 0, len(tr) / np.maximum(counts, 1), 0)
    # sum, not mean: with one bag per step a weighted mean divides w[y] back out
    lossf = nn.CrossEntropyLoss(weight=torch.tensor(cw / cw[counts > 0].mean(), dtype=torch.float32),
                                label_smoothing=args.ls, reduction="sum")
    for ep in range(args.epochs):
        model.train()
        for i, r in enumerate(tr.sample(frac=1, random_state=int(rng.integers(1 << 31))).itertuples()):
            f, b = data[r.image_id]
            x = f[rng.integers(len(f))]
            if not r.is_tma and rng.random() < args.p_block:
                x = x[torch.from_numpy(b == rng.choice(np.unique(b)))]
            elif not r.is_tma:
                x = x[torch.from_numpy(rng.permutation(len(x))[:args.max_tiles])]
            x = x[torch.from_numpy(rng.random(len(x)) > args.tile_drop)] if len(x) > 8 else x
            w = r.w
            (lossf(model(x.float())[None], torch.tensor([r.y])) * w / args.accum).backward()
            if (i + 1) % args.accum == 0:
                opt.step(), opt.zero_grad(), sched.step()
    model.eval()
    out = {}
    with torch.no_grad():
        for r in va.itertuples():
            whole, blocks = bags(*data[r.image_id], r.is_tma)
            out[r.image_id] = (model(whole).softmax(-1).numpy(),
                               np.stack([model(x).softmax(-1).numpy() for x in blocks]))
    return model, out


def score(df, oof):
    p = np.stack([oof[i][0] for i in df.image_id])
    res = {}
    for name, m in [("all", np.ones(len(df), bool)), ("wsi", ~df.is_tma.values), ("tma", df.is_tma.values)]:
        if m.any():
            res[name] = balanced_accuracy_score(df.y[m], p[m].argmax(1))
    wsi = df[~df.is_tma]
    yb = np.concatenate([[y] * len(oof[i][1]) for i, y in zip(wsi.image_id, wsi.y)])
    pb = np.concatenate([oof[i][1] for i in wsi.image_id])
    res["pseudo_tma"] = balanced_accuracy_score(yb, pb.argmax(1))
    return res


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--name", required=True)
    ap.add_argument("--feats", default="feats/phikon-v2")
    ap.add_argument("--views", default="0", help="feature views to sample from at train time (val uses view 0)")
    ap.add_argument("--csv", default="data/train.csv")
    ap.add_argument("--folds", type=int, default=5)
    ap.add_argument("--seeds", default="0")
    ap.add_argument("--epochs", type=int, default=40)
    ap.add_argument("--lr", type=float, default=2e-4)
    ap.add_argument("--wd", type=float, default=1e-2)
    ap.add_argument("--drop", type=float, default=0.25)
    ap.add_argument("--ls", type=float, default=0.1)
    ap.add_argument("--accum", type=int, default=1)  # 16 starved the optimizer: 0.26 vs 0.73 bacc on 140 slides
    ap.add_argument("--p_block", type=float, default=0.5)
    ap.add_argument("--max_tiles", type=int, default=512)
    ap.add_argument("--tile_drop", type=float, default=0.1)
    ap.add_argument("--tma_w", type=float, default=2.0)
    ap.add_argument("--ext", default="", help="external TMA sources from data/ext.csv added to every train fold, e.g. tad,usb")
    ap.add_argument("--ext_w", type=float, default=1.0)
    args = ap.parse_args()
    torch.set_num_threads(4)

    df = pd.read_csv(args.csv)
    views = [int(v) for v in args.views.split(",")]
    df = df[[all((Path(args.feats) / f"v{v}" / f"{i}.npz").exists() for v in views) for i in df.image_id]]
    df = df.reset_index(drop=True)
    df["y"] = df.label.map(CLASSES.index)
    data = load(args.feats, df, views)
    df["w"] = np.where(df.is_tma, args.tma_w, 1.0)
    ext = pd.read_csv("data/ext.csv") if args.ext else pd.DataFrame(columns=["image_id", "label", "src"])
    ext = ext[ext.src.isin(args.ext.split(","))]
    ext = ext.loc[np.array([(Path(args.feats) / "v0" / f"{i}.npz").exists() for i in ext.image_id], bool)]
    ext = ext.assign(is_tma=True, w=args.ext_w, y=ext.label.map(CLASSES.index))
    data.update(load(args.feats, ext, [0]))  # ext: plain view only
    out_dir = Path("runs") / args.name
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"{len(df)} slides ({df.is_tma.sum()} TMA) + {len(ext)} external TMAs", flush=True)

    results = {}
    for seed in map(int, args.seeds.split(",")):
        oof = {}
        skf = StratifiedKFold(args.folds, shuffle=True, random_state=seed)
        efolds = list(skf.split(ext, ext.label)) if len(ext) else [([], [])] * args.folds  # ext OOF = other-hospital TMA proxy
        for fold, ((ti, vi), (eti, evi)) in enumerate(zip(skf.split(df, df.label + df.is_tma.astype(str)), efolds)):
            model, out = run_fold(args, data, pd.concat([df.iloc[ti], ext.iloc[eti]]) if len(ext) else df.iloc[ti],
                                  pd.concat([df.iloc[vi], ext.iloc[evi]]) if len(ext) else df.iloc[vi], seed * 100 + fold)
            oof.update(out)
            torch.save({"model": model.state_dict(), "args": vars(args), "d_in": model.fc[0].in_features,
                        "n_cls": len(CLASSES)},
                       out_dir / f"s{seed}_f{fold}.pt")
            print(f"seed {seed} fold {fold}: " + " ".join(f"{k} {v:.3f}" for k, v in score(df.iloc[vi], out).items()),
                  flush=True)
        results[seed] = score(df, oof)
        for src, g in ext.groupby("src"):
            results[seed][f"ext_{src}"] = balanced_accuracy_score(g.y, [oof[i][0].argmax() for i in g.image_id])
        if len(ext):
            o = ext.y == CLASSES.index("Other")
            results[seed]["ext_other_recall"] = np.mean([oof[i][0].argmax() == 5 for i in ext.image_id[o]])
            results[seed]["ext_other_fp"] = np.mean([oof[i][0].argmax() == 5 for i in ext.image_id[~o]])
        np.save(out_dir / f"oof_s{seed}.npy", oof, allow_pickle=True)
        print(f"seed {seed} OOF: " + " ".join(f"{k} {v:.3f}" for k, v in results[seed].items()), flush=True)
    json.dump({"args": vars(args), "oof": results}, open(out_dir / "result.json", "w"), indent=1)


if __name__ == "__main__":
    main()

In [ ]:
!python train.py --name base --seeds 0,1,2
!python train.py --name ext_all --seeds 0,1,2 --ext tad,usb,hpa,stan
!python train.py --name ph_base --seeds 0,1,2 --feats feats/phikon   # second encoder

## 5. Kaggle inference

`infer.py` runs offline, using two private datasets:
- `ubc-models`: the code plus the `runs/*/s*_f*.pt` checkpoints. It is small and gets a new version per submission.
- `ubc-encoders`: the encoder weights plus wheels, uploaded once.

**Steps:**
- **TMA detection:** test.csv has no `is_tma`, so an image counts as a TMA when `max(w, h) < 6000`. Train TMAs are at most ~4k px and the smallest train WSI is 7.6k px.
- **Pipelining:** three CPU threads tile upcoming images while the GPU embeds the current one. At most six results are in flight, so RAM stays flat.
  Threads are used, not processes: a fork after CUDA init hung the first Kaggle run, and libvips and cv2 release the GIL anyway.
- **Ensembling:** every checkpoint uses the encoder its features came from, so several encoders can be mixed. Probabilities are averaged, and 5-class runs are padded with Other = 0.
- **"Other":** predicted when it is the argmax, or when entropy exceeds `TAU`, which is off by default.
- **Speed:** a WSI takes about 14 s to tile and 18 s to embed on a T4, overlapped, plus ~110 s of startup.

**Python 3.13 pitfall:** Kaggle's image now runs Python 3.13, and pyvips 2.1 fails there at import with `undefined type name VipsCallbackFn`.
The fix is a pyvips 3.2.0 wheel built from the sdist (pure Python, ships in `ubc-models`) plus the `pyvips_binary` 8.18.7 manylinux wheel, which bundles libvips.
The notebook's first cell installs both with `--no-index`.

In [ ]:
%%writefile infer.py
"""Kaggle submission: tile (same code as train) -> encoder features -> ABMIL ensemble -> label, "Other" if entropy > TAU.

Runs offline. Inputs are found by globbing /kaggle/input (mount paths vary):
  ubc-models dataset:   prep.py / feats.py / train.py + runs/*/s*_f*.pt (small, re-uploaded often)
  ubc-encoders dataset: encoder weight dirs (phikon-v2/, ...) + wheels (uploaded once)
Each MIL run uses the encoder its features came from (args.feats = feats/<encoder>).
Env overrides for local testing: TEST_DIR, TEST_CSV, MODELS, ENCODERS, OUT, TAU, WORKERS, BLOCKS.
"""
import os
import sys
import time
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch


def find(pattern):
    return next(Path("/kaggle/input").glob(pattern))


TEST_DIR = Path(os.environ.get("TEST_DIR") or find("**/test_images"))
TEST_CSV = os.environ.get("TEST_CSV") or TEST_DIR.parent / "test.csv"
MODELS = Path(os.environ.get("MODELS") or find("**/runs").parent)
ENCODERS = Path(os.environ.get("ENCODERS") or find("**/phikon-v2").parent)
OUT = os.environ.get("OUT", "submission.csv")
TAU = float(os.environ.get("TAU", "inf"))  # entropy above this -> Other; tuned on late-submission LB
WORKERS = int(os.environ.get("WORKERS", "3"))
BLOCKS = int(os.environ.get("BLOCKS", "48"))  # WSI tile budget: lower it if the run times out
sys.path.insert(0, str(MODELS))
from feats import MEAN, STD, encoder  # noqa: E402
from prep import tile_image  # noqa: E402
from train import ABMIL, CLASSES  # noqa: E402


def tiles(image_id, w, h):
    """-> (id, tma, [jpeg bytes], seconds). JPEG keeps in-flight RAM ~7x smaller than pixels and matches train's round-trip."""
    tma = max(w, h) < 6000  # train: TMA <= ~4k px, smallest WSI 7.6k px; test.csv has no is_tma
    t = time.time()
    jpgs = [j for b in tile_image(TEST_DIR / f"{image_id}.png", tma, BLOCKS, 0) for *_, j in b]
    return image_id, tma, jpgs, time.time() - t


def decode(jpgs):
    return np.stack([cv2.cvtColor(cv2.imdecode(np.frombuffer(j, np.uint8), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
                     for j in jpgs])


@torch.no_grad()
def main():
    t0 = time.time()
    df = pd.read_csv(TEST_CSV)
    encs, mils = {}, []  # encoder name -> model; [(encoder name, MIL)]
    for p in sorted(MODELS.glob("runs/*/s*_f*.pt")):
        ck = torch.load(p, map_location="cpu", weights_only=False)
        name = Path(ck["args"]["feats"]).name
        if name not in encs:
            encs[name] = encoder(ENCODERS / name).cuda().eval()
        m = ABMIL(d_in=ck["d_in"], n_cls=ck.get("n_cls", 5), drop=ck["args"]["drop"])
        m.load_state_dict(ck["model"])
        mils.append((name, m.cuda().eval()))
    print(f"{len(df)} images, {len(mils)} MIL models on {list(encs)}", flush=True)
    mean, std = torch.tensor(MEAN).cuda(), torch.tensor(STD).cuda()
    rows, todo, futs = {}, iter(df.itertuples()), {}
    # CPU tiles while the GPU embeds. At most 2*WORKERS results in flight so RAM stays flat (a WSI is ~300 MB of pixels).
    # threads, not processes: libvips/cv2 release the GIL, and fork after CUDA init hung the first Kaggle run
    with ThreadPoolExecutor(WORKERS) as ex:
        def submit():
            r = next(todo, None)
            if r is not None:
                futs[ex.submit(tiles, r.image_id, r.image_width, r.image_height)] = r.image_id

        for _ in range(2 * WORKERS):
            submit()
        while futs:
            for f in wait(futs, return_when=FIRST_COMPLETED).done:
                iid = futs.pop(f)
                submit()
                try:
                    _, tma, jpgs, t_tile = f.result()
                    t_emb = time.time()
                    feats = {n: [] for n in encs}
                    for s in range(0, len(jpgs), 128):
                        xb = ((torch.from_numpy(decode(jpgs[s:s + 128])).cuda().float() - mean) / std).permute(0, 3, 1, 2)
                        with torch.autocast("cuda", dtype=torch.float16):
                            for n, enc in encs.items():
                                feats[n].append(enc(xb).float())
                    feats = {n: torch.cat(v) for n, v in feats.items()}
                    p = torch.stack([torch.nn.functional.pad(m(feats[n]).softmax(-1), (0, len(CLASSES) - m.head.out_features))
                                     for n, m in mils]).mean(0).cpu().numpy()  # 5-class runs get Other = 0
                    ent = float(-(p * np.log(p + 1e-9)).sum())
                    rows[iid] = "Other" if ent > TAU else CLASSES[p.argmax()]
                    print(f"[{len(rows)}/{len(df)}] {iid} tma={tma} tiles={len(jpgs)} {rows[iid]} p={p.round(2)} "
                          f"H={ent:.2f} tile={t_tile:.0f}s emb={time.time() - t_emb:.0f}s t={time.time() - t0:.0f}s", flush=True)
                except Exception as e:  # never lose the whole submission to one bad image
                    print("FAILED", iid, repr(e), flush=True)
    sub = pd.DataFrame({"image_id": df.image_id, "label": [rows.get(i, "HGSC") for i in df.image_id]})
    sub.to_csv(OUT, index=False)
    print(sub.label.value_counts().to_dict(), f"{time.time() - t0:.0f}s", flush=True)


if __name__ == "__main__":
    main()

`pack.sh` copies the code and the chosen runs into `kaggle/models`, versions the `ubc-models` dataset, and rebuilds the kernel notebook: a pip cell, then the env overrides, then `infer.py`.
Then push the kernel, and submit its output once it completes.
A submission can show COMPLETE with blank scores when it actually failed, so check `error_description`.

In [ ]:
%%writefile pack.sh
#!/bin/bash
# On .51: bundle code + MIL runs into the ubc-models dataset, rebuild the notebook from infer.py, push both.
# usage: bash pack.sh "message" RUN [RUN ...] [ENV=VAL ...]   (runs from ~/ubc/runs; ENV pairs go into the notebook)
set -e
cd ~/ubc
msg=$1; shift
runs=(); envs=()
for a in "$@"; do [[ $a == *=* ]] && envs+=("$a") || runs+=("$a"); done
K=kaggle/models
rm -rf $K/runs; mkdir -p $K/runs
cp prep.py feats.py train.py kaggle/wheels/pyvips-3.2.0-py3-none-any.whl $K/  # Kaggle is py3.13: pyvips 2.1 breaks
for r in "${runs[@]}"; do mkdir -p $K/runs/$r; cp runs/$r/s*_f*.pt runs/$r/result.json $K/runs/$r/; done
.venv/bin/python - "${envs[@]}" <<'EOF'
import json, sys
env = dict(a.split("=", 1) for a in sys.argv[1:])
cell = lambda s: {"cell_type": "code", "metadata": {}, "execution_count": None, "outputs": [], "source": s}
nb = {"nbformat": 4, "nbformat_minor": 4,
      "metadata": {"kernelspec": {"name": "python3", "display_name": "Python 3", "language": "python"}},
      "cells": [cell('!pip install -q --no-index --no-deps $(find /kaggle/input -name "pyvips-3*.whl" -o -name "pyvips_binary*.whl")'),
                cell(f"import os\nos.environ.update({env!r})\n\n" + open("infer.py").read())]}
json.dump(nb, open("kaggle/kernel/infer.ipynb", "w"), indent=1)
EOF
if .venv/bin/kaggle datasets status yashd3/ubc-models >/dev/null 2>&1; then
  .venv/bin/kaggle datasets version -p $K -m "$msg" -r tar
else
  .venv/bin/kaggle datasets create -p $K -r tar
fi
echo "dataset pushed; wait for it to be ready, then: .venv/bin/kaggle kernels push -p kaggle/kernel"

In [ ]:
!bash pack.sh "base + ext_all, 3 seeds each" base ext_all   # run names under runs/
!kaggle kernels push -p kaggle/kernel --accelerator NvidiaTeslaT4
!kaggle competitions submit ubc-ocean -k <user>/ubc-infer -v <version> -f submission.csv -m "base + ext_all"

## What would improve this next

- **Encoder ensemble:** Phikon (ViT-B) features alongside Phikon-v2. Phikon is ViT-B, so it adds roughly a third of Phikon-v2's embedding time.
- **A stronger encoder:** UNI or another model pretrained on more pathology data. Top solutions relied on these, but access is gated.
- **Better "Other" detection:** more outlier tissue, or a tile-level outlier score. The entropy threshold here was too blunt.
- **Validation that tracks the LB:** the external split leaks each vendor into both train and validation, so 0.75 on external TMAs is optimistic.
  Leave-one-source-out validation would be a fairer estimate of unseen hospitals.